# ClinGuard-DiLoCo: Kaggle session 3 (gate G3 retry: can the model learn findings?)

Session 2 showed that site A's fine-tuned model writes the same "normal" report for almost every study: it
scores exactly like a model that always writes "normal". This session tests the likely causes, one setting
at a time, against an anchor run (site A, learning rate 2e-4):

| GPU | Jobs, in order | What it tests | Estimate |
|---|---|---|---|
| 0 | `g3_A_lr2e4`, then `g3_A_lr5e4` | learning rate (2e-4 anchor; 5e-4) | ~3 h |
| 1 | `g3_A_mlp_lr2e4`, then `g3_B_lr2e4` | more LoRA capacity (MLP layers); training on site B, where abnormal studies are common | ~3 h |
| 0 | image test, after both GPUs finish | does each model's prediction depend on the X-ray? | ~15 min |

Every round is uploaded to the private HF repo with a lease, so a crash or timeout loses at most ~15 min:
run the notebook again and each job continues where it stopped.

**Settings** (right-hand panel): GPU **T4 x2**, Internet **on**, input dataset **raddar/chest-xrays-indiana-university**,
secret **HF_TOKEN** attached. Keep the notebook **private**.

**How to run:** *Save Version* -> **Save & Run All (Commit)**, then close the tab: it keeps running (up to 12 h)
and the last cell prints a summary. Open the finished version's *Logs* or *Output* and send the last two cells.
Running cells by hand in the editor stops when the browser session ends.

In [ ]:
HF_REPO = "tasbid66/clinguard-runs"
GIT_REF = "main"     # a resumed run refuses a different commit; to resume after main moved, set the commit printed in its log
import glob, os, subprocess, sys, time, json
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!rm -rf /kaggle/working/ClinGuard && git clone -q https://github.com/tasbidrahman10/ClinGuard.git /kaggle/working/ClinGuard
%cd /kaggle/working/ClinGuard
!git checkout -q {GIT_REF} && git log --oneline -1
!pip install -q "transformers==5.23.0" "peft==0.22.0" "accelerate==1.16.0" bitsandbytes pyyaml
!pip install -q --no-deps -e .
src = os.path.dirname(glob.glob("/kaggle/input/**/indiana_reports.csv", recursive=True)[0])
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/iu_xray"):
    os.symlink(src, "data/raw/iu_xray")
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

Download the 3B model once, so the two GPUs do not race for it.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/ClinGuard/src")
from huggingface_hub import snapshot_download
from clinguard.model import load
snapshot_download(load.MODEL_ID, revision=load.MODEL_REVISION)
print("model cached")

## Launch the training jobs (one queue per GPU, in the background)

In [ ]:
import os, subprocess
HF_REPO = "tasbid66/clinguard-runs"
os.chdir("/kaggle/working/ClinGuard")
os.makedirs("/kaggle/working/logs", exist_ok=True)
RUN = f"python scripts/run_job.py --hf-repo {HF_REPO} --work-dir /kaggle/working/runs --config configs/experiment"
queues = {0: ["g3_A_lr2e4", "g3_A_lr5e4"], 1: ["g3_A_mlp_lr2e4", "g3_B_lr2e4"]}
procs = {}
for gpu, names in queues.items():
    cmd = " ; ".join(f"{RUN}/{n}.yaml" for n in names)     # ';' so the second job still runs if the first fails
    log = open(f"/kaggle/working/logs/gpu{gpu}.log", "a")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    procs[gpu] = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT, env=env)
    print("GPU", gpu, "started:", " -> ".join(names))

## Wait, printing progress every 5 minutes

In [ ]:
import time
def tail(path, n=3):
    lines = [l for l in open(path, errors="replace").read().splitlines()
             if l.strip() and "Warning" not in l and "warn" not in l]
    return lines[-n:]
start = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(f"--- {int(time.time() - start) // 60} min")
    for gpu in procs:
        state = "running" if procs[gpu].poll() is None else f"exit {procs[gpu].returncode}"
        print(f"GPU {gpu} ({state}):", *tail(f"/kaggle/working/logs/gpu{gpu}.log"), sep="\n   ")
print({gpu: p.returncode for gpu, p in procs.items()})

## Image test (does each model use the X-ray?)

In [ ]:
import os, subprocess
HF_REPO = "tasbid66/clinguard-runs"
os.chdir("/kaggle/working/ClinGuard")
names = ["s1_single_site_A", "g3_A_lr2e4", "g3_A_lr5e4", "g3_A_mlp_lr2e4", "g3_B_lr2e4"]
cmd = (f"python scripts/image_test.py --hf-repo {HF_REPO} --name session3 --site A --work-dir /kaggle/working/runs "
       "--configs " + " ".join(f"configs/experiment/{n}.yaml" for n in names))
r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
open("/kaggle/working/logs/image_test.log", "w").write(r.stdout + r.stderr)
print("\n".join(l for l in r.stdout.splitlines() if "Warning" not in l)[-4000:])
if r.returncode:
    print("image test failed:", r.stderr[-3000:])

## Summary (self-contained: it can be run on its own after a restart, once cell 2 has run)

In [ ]:
import importlib.util, json, os
from huggingface_hub import hf_hub_download
HF_REPO = "tasbid66/clinguard-runs"
os.chdir("/kaggle/working/ClinGuard")
spec = importlib.util.spec_from_file_location("rj", "scripts/run_job.py"); rj = importlib.util.module_from_spec(spec); spec.loader.exec_module(rj)
def fetch(path):
    try:
        return hf_hub_download(HF_REPO, path, force_download=True)
    except Exception as e:
        print("missing:", path, type(e).__name__); return None
for n in ("g3_A_lr2e4", "g3_A_lr5e4", "g3_A_mlp_lr2e4", "g3_B_lr2e4"):
    rid = rj.run_id_for(rj.load_config(f"configs/experiment/{n}.yaml"))
    lease = fetch(f"runs/{rid}/LEASE.json")
    print(f"{n:16s} {rid}  status: {json.load(open(lease))['status'] if lease else '-'}")
    r = fetch(f"runs/{rid}/rounds.jsonl")
    if r:
        rows = [json.loads(l) for l in open(r)]
        site = next(iter(rows[0]["train_loss"]))
        print("   train loss by round:", " ".join(f"{x['train_loss'][site]:.3f}" for x in rows),
              f"| {rows[-1]['cost'][site]['train_sec'] / 100:.2f} s/step")
t = fetch("diagnostics/image_test_session3.json")
if t:
    t = json.load(open(t))
    print(f"\nimage test, site {t['site']} {t['role']}, {t['n']} studies (loss of the reference report; lower = better)")
    for label, r in t["results"].items():
        a, b = r["all"], r["abnormal"]
        print(f"  {label[:52]:52s} true {a['loss_true']:.3f} other {a['loss_shuffled']:.3f} blank {a['loss_blank']:.3f}"
              f" | abnormal: true {b['loss_true']:.3f} other {b['loss_shuffled']:.3f}"
              f" true better in {b['share_true_image_better']:.0%}")

**Send Tasbid / Claude:** the output of the last two cells. Generated reports stay in the HF repo; they are
scored locally in the evaluation environment.